# Cambridge sedation EEG: ingestion and within-subject analysis

Public archive of Chennu et al. (2016), used here as an EEGLAB epoch dataset with a repeated-measures condition structure.

The source paper describes a 0.5–45 Hz filter and an average reference. The file header inspected here can still say high-pass 0 Hz, low-pass 125 Hz, and no reference label. This notebook does not band-pass or re-reference the files. It uses the samples as stored. Condition labels are read from `datainfo.mat` in this notebook. The library does not know them. Licence, DOI, archive size, and the header-versus-paper distinction are in `docs/validation/DATASETS.md`. The Cambridge archive is about 3.7 GB and is not part of the package.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.io
from IPython.display import display
from eegmath import extract_features, load
from eegmath.preprocess import qc_summary
from eegmath.signal import adf_stationarity, morlet_power
from eegmath.spectral import welch_psd
from eegmath.stats import compare_groups, design_summary, repeated_measures
from eegmath.viz import plot_forest, plot_psd

def _validation_data() -> Path:
    override = os.environ.get('EEGMATH_DATA')
    if override:
        return Path(override).expanduser()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'external_data').is_dir():
            return candidate / 'external_data'
    return here / 'external_data'

DATA = _validation_data()
ROOT = DATA / 'cambridge_propofol' / 'extracted' / 'Sedation-RestingState'
def _validation_output() -> Path:
    override = os.environ.get('EEGMATH_VALIDATION_OUT')
    if override:
        return Path(override).expanduser()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'notebooks').is_dir():
            return candidate / 'validation_output'
    return here / 'validation_output'

OUT = _validation_output()
OUT.mkdir(parents=True, exist_ok=True)
assert ROOT.joinpath('datainfo.mat').exists(), (
    f'Cambridge files were not found at {ROOT}. Download them with '
    'scripts/fetch_validation_data.py, or set EEGMATH_DATA to the directory '
    'that contains cambridge_propofol/. See docs/validation/DATASETS.md.'
)
LEVELS = {1: 'baseline', 2: 'mild', 3: 'moderate', 4: 'recovery'}
CHANNELS = ['O1', 'O2', 'Oz']


## One recording

In [2]:
example_path = sorted(ROOT.glob('*.set'))[0]
example = load(example_path)
summary = example.describe()
summary['reader_note'] = example.metadata.get('reader_note')
summary['highpass_hz'] = example.metadata.get('highpass')
summary['lowpass_hz'] = example.metadata.get('lowpass')
display(pd.Series(summary))
qc = qc_summary(example)
display(pd.Series({k: qc[k] for k in ('median_channel_std', 'amplitude_scale_hint', 'n_channels', 'duration_sec')}))
assert example.epoched and example.sfreq == 250 and example.units == 'V'
assert not any('bandpass' in note or 'rereference' in note for note in example.provenance)
assert qc['amplitude_scale_hint'] == 'consistent_with_volts'

recording_id                                02-2010-anest 20100210 135.003
subject_id                                                            None
session_id                                                            None
condition                                                             None
epoched                                                               True
n_epochs                                                                39
n_channels                                                              91
n_times                                                               2500
sfreq                                                                250.0
duration_sec                                                          10.0
units                                                                    V
reference                                                             None
n_positioned_channels                                                   91
tmin                     

median_channel_std                   0.000007
amplitude_scale_hint    consistent_with_volts
n_channels                                 91
duration_sec                             10.0
dtype: object

## Features of that recording

Spectral columns come from the epochs as stored. A 10-second epoch has enough 2-second Welch segments for a spectrum and not, by itself, 20 connectivity segments, so connectivity uses a 1-second segment and pools epochs. The figure is the mean of those epoch spectra. Resting epochs do not share an event time, so the spectrum of their averaged voltage is a different quantity.


In [3]:
posterior = example.pick_channels(CHANNELS)
spectral = extract_features(posterior, 'spectral', reduce_epochs=True)
display(spectral[['channel', 'alpha_relative', 'alpha_peak_freq', 'sef50', 'median_freq', 'spectral_entropy']].round(4))
relative_cols = [c for c in spectral.columns if c.endswith('_relative')]
assert np.allclose(spectral[relative_cols].sum(axis=1), 1.0, atol=1e-5)
assert np.allclose(spectral['median_freq'], spectral['sef50'])
freqs, epoch_psd = welch_psd(posterior.data, posterior.sfreq)
psd = epoch_psd.mean(axis=0)
oz = psd[list(posterior.ch_names).index('Oz')]
alpha_density = float(np.mean(oz[(freqs >= 8) & (freqs < 13)]))
theta_density = float(np.mean(oz[(freqs >= 4) & (freqs < 8)]))
beta_density = float(np.mean(oz[(freqs >= 20) & (freqs < 30)]))
assert alpha_density > theta_density and alpha_density > beta_density
fig, ax = plot_psd(freqs, psd, ch_names=list(posterior.ch_names), fmax=45)
fig.savefig(OUT / 'cambridge_example_psd.png', dpi=120, bbox_inches='tight')
fig.clf()


,channel,alpha_relative,alpha_peak_freq,sef50,median_freq,spectral_entropy
0,O1,0.5082,10.5513,8.4873,8.4873,0.6714
1,O2,0.6855,10.4487,10.1721,10.1721,0.6672
2,Oz,0.6306,10.7564,10.0776,10.0776,0.6914


<Figure size 700x400 with 0 Axes>

In [4]:
one = example.spawn('first-epoch', data=example.data[0:1], ch_names=example.ch_names)
picked = one.pick_channels(['Oz'])
time_features = extract_features(picked, ['permutation_entropy', 'hjorth_mobility', 'line_length', 'rms'])
shown = time_features[['permutation_entropy', 'hjorth_mobility', 'line_length', 'rms']].copy()
shown['rms_uV'] = shown['rms'] * (1e6 if picked.units == 'V' else 1.0)
shown['line_length_uV_per_s'] = shown['line_length'] * (1e6 if picked.units == 'V' else 1.0)
display(shown[['permutation_entropy', 'hjorth_mobility', 'rms_uV', 'line_length_uV_per_s']].round(4))
assert np.isfinite(time_features[['permutation_entropy', 'hjorth_mobility']].to_numpy()).all()
assert float(shown['rms_uV'].iloc[0]) > 0.1
stationarity = adf_stationarity(picked.data[0, 0])
display(pd.Series({k: stationarity[k] for k in ('statistic', 'p_value', 'null', 'note')}))
power, times = morlet_power(picked, np.array([4.0, 10.0, 20.0]), n_cycles=5)
assert power.shape[2] == 3 and len(times) == picked.n_times

,permutation_entropy,hjorth_mobility,rms_uV,line_length_uV_per_s
0,0.6431,64.8401,10.3447,536.2604


statistic                                            -7.079939
p_value                                                    0.0
null                                                 unit root
note         Rejecting a unit root is not a general demonst...
dtype: object

In [5]:
from eegmath.connectivity import connectivity_matrices
nodes = [name for name in ('F3', 'F4', 'P3', 'P4', 'O1', 'O2', 'Fz', 'Pz') if name in example.ch_names][:6]
assert len(nodes) >= 4, example.ch_names
matrices, meta = connectivity_matrices(
    example.pick_channels(nodes).data, example.sfreq,
    metrics=('coherence', 'wpli'), min_segments=10, segment_sec=1.0,
)
print('segments', meta['n_segments'], 'sufficient', meta['sufficient_segments'])
assert meta['sufficient_segments']
assert np.isfinite(matrices['coherence']['alpha']).all()
assert matrices['coherence']['alpha'].shape == (len(nodes), len(nodes))

segments 741 sufficient True


## Statistics on a table this library did not have to create

The outcome column is just a column. The subject column is required because the default unit of `compare_groups` is the subject.

In [6]:
rng = np.random.default_rng(0)
external = pd.DataFrame({
    'subject_id': [f's{i:02d}' for i in range(12)] * 2,
    'condition': ['pre'] * 12 + ['post'] * 12,
    'custom_score': np.r_[rng.normal(0.0, 0.4, 12), rng.normal(0.8, 0.4, 12)],
})
external_result = compare_groups(
    external, ['custom_score'], 'condition', 'post', 'pre',
    subject='subject_id', test='wilcoxon',
)
display(external_result[['feature', 'n_a', 'effect', 'p_value', 'ci_low', 'ci_high', 'aggregation']])
assert external_result.loc[0, 'n_a'] == 12
assert external_result.loc[0, 'aggregation'] == 'mean_within_subject_and_group'

,feature,n_a,effect,p_value,ci_low,ci_high,aggregation
0,custom_score,12,1.3504,0.001465,0.365898,1.016167,mean_within_subject_and_group


## All recordings, one occipital summary per file

The subject-level value is the mean of the channel-wise relative powers at O1, O2, and Oz. `aggregate_regions` rebuilds a ratio from averaged absolute power and is a different summary.


In [7]:
def datainfo(path: Path) -> pd.DataFrame:
    raw = scipy.io.loadmat(path, squeeze_me=False)['datainfo']
    rows = []
    for row in raw:
        values = [np.asarray(cell).reshape(-1)[0].item() for cell in row]
        rows.append({
            'recording_id': str(values[0]),
            'level': int(values[1]),
            'plasma': float(values[2]),
            'reaction_time': float(values[3]),
            'correct_responses': float(values[4]),
        })
    frame = pd.DataFrame(rows)
    frame['condition'] = frame['level'].map(LEVELS)
    frame['subject_id'] = frame['recording_id'].str.slice(0, 2)
    return frame

info = datainfo(ROOT / 'datainfo.mat')
assert set(info['condition']) == set(LEVELS.values())
assert info.groupby('condition').size().min() == 20
files = {path.stem: path for path in ROOT.glob('*.set')}
assert set(info['recording_id']) == set(files)
rows = []
for i, record in enumerate(info.itertuples(index=False), start=1):
    if i == 1 or i % 20 == 0:
        print(f'file {i}/{len(info)}', flush=True)
    recording = load(files[record.recording_id]).spawn(
        'condition-label', subject_id=record.subject_id, condition=record.condition,
    )
    table = extract_features(
        recording.pick_channels(CHANNELS),
        ['alpha_relative', 'alpha_power', 'theta_alpha_ratio'],
        reduce_epochs=True,
    )
    rows.append(table[['subject_id', 'condition', 'channel', 'alpha_relative', 'alpha_power', 'theta_alpha_ratio']])
occipital = pd.concat(rows, ignore_index=True)
subject = (
    occipital.groupby(['subject_id', 'condition'], as_index=False)[['alpha_relative', 'alpha_power', 'theta_alpha_ratio']]
    .mean()
    .merge(info[['subject_id', 'condition', 'plasma', 'reaction_time', 'correct_responses']], on=['subject_id', 'condition'])
)
assert subject.groupby('subject_id')['condition'].nunique().min() == 4
assert len(subject) == 80
subject.to_csv(OUT / 'cambridge_occipital_subject_means.csv', index=False)
display(design_summary(subject, subject='subject_id', condition='condition'))
display(subject.groupby('condition')['alpha_relative'].agg(['mean', 'std', 'count']).round(4))

file 1/80


file 20/80


file 40/80


file 60/80


file 80/80


{'n_rows': 80,
 'n_subjects': 20,
 'rows_per_subject': {'count': 20.0,
  'mean': 4.0,
  'std': 0.0,
  'min': 4.0,
  '25%': 4.0,
  '50%': 4.0,
  '75%': 4.0,
  'max': 4.0},
 'conditions': {'baseline': 20, 'mild': 20, 'moderate': 20, 'recovery': 20},
 'max_rows_per_subject_condition': 1,
 'subjects_in_multiple_conditions': 20,
 'complete_subjects': 20}

,mean,std,count
condition,,,
baseline,0.4922,0.1658,20
mild,0.4692,0.1782,20
moderate,0.2986,0.1436,20
recovery,0.5511,0.1050,20


In [8]:
fit = repeated_measures(
    subject, 'alpha_relative', 'condition', 'subject_id',
    reference='baseline', pairs=[('baseline', 'moderate')],
)
display(fit.coefficients.round(4))
display(pd.Series(fit.omnibus))
display(fit.contrasts.round(4))
assert fit.n_subjects == 20
assert fit.converged
paired = compare_groups(
    subject, ['alpha_relative', 'theta_alpha_ratio'], 'condition', 'moderate', 'baseline',
    subject='subject_id', test='wilcoxon',
)
display(paired[['feature', 'n_a', 'effect', 'mean_difference_a_minus_b', 'ci_low', 'ci_high', 'p_value', 'p_corrected']])
assert paired['n_a'].eq(20).all()
fig, ax = plot_forest(fit.contrasts, estimate='estimate', low='ci_low', high='ci_high', label='contrast')
fig.savefig(OUT / 'cambridge_alpha_contrast.png', dpi=120, bbox_inches='tight')
fig.clf()
association = subject.loc[subject['condition'] == 'moderate', ['alpha_relative', 'reaction_time']].dropna()
from eegmath.stats import correlation
display(pd.Series(correlation(association['reaction_time'], association['alpha_relative'], method='spearman')))

,term,estimate,se,statistic,df,p_value,ci_low,ci_high
0,Intercept,0.4922,0.0337,14.5998,16.0,0.0000,0.4207,0.5636
1,C(Q('condition'))[T.mild],-0.0230,0.0292,-0.7878,16.0,0.4424,-0.0849,0.0389
2,C(Q('condition'))[T.moderate],-0.1936,0.0292,-6.6270,16.0,0.0000,-0.2555,-0.1317
3,C(Q('condition'))[T.recovery],0.0590,0.0292,2.0190,16.0,0.0606,-0.0029,0.1209


statistic                                            53.774659
df                                                           3
p_value                                                    0.0
converged                                                 True
note         Both models were refit with ML because a REML ...
dtype: object

,a,b,contrast,estimate,se,statistic,df,p_value,ci_low,ci_high
0,baseline,moderate,moderate - baseline,-0.1936,0.0292,-6.627,16.0,0.0,-0.2555,-0.1317


,feature,n_a,effect,mean_difference_a_minus_b,ci_low,ci_high,p_value,p_corrected
0,alpha_relative,20,-1.451635,-0.193572,-0.255980,-0.131163,0.000010,0.000019
1,theta_alpha_ratio,20,0.343617,0.153788,-0.055675,0.363250,0.001432,0.001432


method                      spearman
r                            -0.0258
p_value                      0.91906
n                                 18
ci_low                           NaN
ci_high                          NaN
ci_note    not computed for Spearman
dtype: object

<Figure size 700x300 with 0 Axes>

The mixed-model contrast and the Wilcoxon result are estimates for this occipital summary. A large p-value would still mean the functions ran the tests that were named. The notebook checks the design counts, convergence, and that ratios stay inside the partition.